<h3 style="color:#1E3A8A; font-family:Arial, Helvetica, sans-serif; margin-bottom:6px;">
  <b>IST 691 — Phase 2 (Part 1): Baseline Embedding Artifact (SentenceTransformer Inference)</b>
</h3>

<div style="font-family:Arial, Helvetica, sans-serif; line-height:1.55;">
  <p style="margin-top:0;">
    <b>Big picture.</b> This module generates the baseline embedding artifact for the Phase 1 refined sentence dataset using a pretrained SentenceTransformer in inference-only mode. The exported artifact defines the reference embedding space and provides a schema-aligned baseline for Phase 2 (Part 3) comparison.
  </p>

  <p>
    <b>Single source of truth (input).</b> One sentence-level CSV is loaded from the local working directory (or an explicit config path). Required fields are:
    <ul style="margin-top:4px;">
      <li><b>belief_sentence</b> (sentence text to embed)</li>
      <li><b>year_quarter</b> (timeline key for downstream aggregation and comparison)</li>
      <li><b>stable identifier</b> via <code>sentence_id</code> / <code>hash_id</code> / <code>id</code>, or a deterministic <code>sentence_id</code> constructed when absent</li>
    </ul>
  </p>

  <p>
    <b>Core logic flow.</b>
    <ol style="margin-top:4px;">
      <li><b>Resolve input</b> CSV path via a deterministic resolution policy (explicit path → expected filename → auto-detect).</li>
      <li><b>Load + validate</b> schema and required columns; preserve row order.</li>
      <li><b>Stabilize identifiers</b> by selecting an existing id column when available, otherwise generating a deterministic <code>sentence_id</code>.</li>
      <li><b>Embed</b> all sentences using <code>sentence-transformers/all-MiniLM-L6-v2</code> (inference-only; no training).</li>
      <li><b>Export</b> standardized artifacts and a run manifest for traceability and downstream reproducibility.</li>
    </ol>
  </p>

  <p>
    <b>Deliverables (local exports).</b>
    <ul style="margin-top:4px;">
      <li><b>Embedding artifact</b>: <code>sentence_id, year_quarter, belief_sentence, embedding</code> (parquet or csv)</li>
      <li><b>Summary table</b>: row counts, runtime, embedding dimension, and norm statistics</li>
      <li><b>Run manifest</b>: input checksum + identifier policy + output file list for audit-ready data-flow</li>
    </ul>
  </p>

  <p style="margin-bottom:0;">
    <b>Console exhibit.</b> The run prints a compact header, resolved input pointers, output path pointers, a single-row summary preview, and a small top-row embedding preview with embedding length only.
  </p>
</div>

<hr style="border:none; border-top:1px solid #ddd; marg


In [ ]:
"""
====================================================================================================
IST 691 — Phase 2 (Part 1)  |  Baseline Embedding Artifact (SentenceTransformer Inference)
====================================================================================================

purpose
  - produce a baseline sentence-embedding artifact from the Phase 1 refined sentence-level dataset
  - establish a stable, schema-aligned embedding table for symmetric comparison in Phase 2 (part 3)

inputs (single source of truth)
  - input csv: phase 1 refined, sentence-level dataset (local working directory or config-pinned path)
  - required columns: belief_sentence, year_quarter
  - identifier policy:
       - use an existing stable id column when present (sentence_id / hash_id / id), otherwise
       - construct sentence_id deterministically from normalized text (collision-safe)

processing (logic flow)
  1) resolve input csv path (explicit path → expected filename → auto-detect candidate csv)
  2) load dataset and validate schema (required columns; row/column counts)
  3) stabilize identifiers (existing id column or deterministic sentence_id construction)
  4) embed all belief_sentence rows using:
       - base checkpoint: sentence-transformers/all-MiniLM-L6-v2
       - inference-only (no supervision; no parameter updates)
       - device policy: mps if available; otherwise cpu
  5) export artifacts + audit manifest + console exhibit (compact previews; path pointers)

outputs (local exports)
  - embeddings artifact: part1_baseline_embeddings.parquet (or .csv)
       - schema: sentence_id, year_quarter, belief_sentence, embedding
  - summary table: part1_baseline_summary.csv
       - row counts, missing-text count, embedding_dim, runtime, embedding norm stats, sentence_id_list_hash
  - run manifest: part1_baseline_embedding_manifest.json
       - input checksum, schema, model parameters, output file list (traceable data-flow)
  - preview table (console) showing top rows with embedding length only

console exhibit (immediate review)
  - run header (timestamp + platform + python + cwd)
  - input pointers (resolved path + md5 checksum)
  - output pointers (artifact + summary + manifest)
  - summary preview (single-row table)
  - embeddings preview (top rows; embedding shown as len=dim)

====================================================================================================

December 2025 | Syracuse University | IST 691 Deep Learning Term Project

Dujun; Yifeng; Isha
"""


from __future__ import annotations

import sys
import json
import time
import hashlib
import platform
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer


# =============================================================================
# config (part 1 only)
# =============================================================================

CFG = {
    # input
    "input_csv_path": None,  # set to an absolute path to hard-pin; otherwise resolved locally
    "expected_filename": "belief_extraction_results_deduped_year_quarter_flattened_repeat_deduped.csv",
    "text_col": "belief_sentence",
    "quarter_col": "year_quarter",
    "preferred_id_cols": ["sentence_id", "hash_id", "id"],

    # model
    "baseline_model_name": "sentence-transformers/all-MiniLM-L6-v2",
    "batch_size": 64,

    # outputs
    "output_root": str(Path.cwd().resolve() / "IST691_phase2_output"),
    "save_format": "parquet",  # parquet preferred; fallback to csv is supported
}


# =============================================================================
# helpers
# =============================================================================

def _header(title: str) -> None:
    bar = "=" * 100
    print(f"\n{bar}\n>>> {title}\n{bar}")

def _section(title: str) -> None:
    bar = "-" * 80
    print(f"\n{bar}\n[ {title} ]\n{bar}")

def _md5_file(path: Path, chunk: int = 1024 * 1024) -> str:
    h = hashlib.md5()
    with path.open("rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def _pick_device() -> str:
    try:
        import torch
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            return "mps"
    except Exception:
        pass
    return "cpu"

def _resolve_input_csv() -> Path:
    cwd = Path.cwd().resolve()

    if CFG["input_csv_path"]:
        p = Path(CFG["input_csv_path"]).expanduser().resolve()
        if not p.exists():
            raise FileNotFoundError(f"input csv not found: {p}")
        return p

    expected = cwd / CFG["expected_filename"]
    if expected.exists():
        return expected

    candidates = sorted(cwd.glob("belief_extraction*.csv"))
    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) > 1:
        candidates = sorted(candidates, key=lambda x: x.stat().st_size, reverse=True)
        return candidates[0]

    candidates = sorted(cwd.glob("*.csv"))
    if len(candidates) == 1:
        return candidates[0]

    raise FileNotFoundError(
        "input csv could not be resolved from local directory.\n"
        f"cwd: {cwd}\n"
        f"expected: {CFG['expected_filename']}\n"
        f"csv files found: {[p.name for p in candidates] if candidates else '(none)'}"
    )

def _pick_id_col(df: pd.DataFrame) -> str | None:
    for c in CFG["preferred_id_cols"]:
        if c in df.columns:
            return c
    return None

def _build_stable_sentence_id(df: pd.DataFrame, text_col: str) -> pd.Series:
    norm = df[text_col].astype(str).str.strip()
    ids = norm.apply(lambda s: hashlib.md5(s.encode("utf-8")).hexdigest()[:16])
    if ids.duplicated().any():
        ids = ids + "_" + df.index.astype(str)
    return ids


# =============================================================================
# part 1
# =============================================================================

def part1_baseline_embedding() -> None:
    t0 = time.time()
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    out_root = Path(CFG["output_root"]).resolve()
    out_dir = out_root / "part1_baseline"
    logs_dir = out_root / "logs"
    manifests_dir = out_root / "manifests"
    for d in [out_dir, logs_dir, manifests_dir]:
        d.mkdir(parents=True, exist_ok=True)

    _header("ist 691 — phase 2 (part 1): baseline embedding")
    _section("inputs")
    print("timestamp:", ts)
    print("platform :", platform.platform())
    print("python   :", sys.version.split()[0])
    print("cwd      :", Path.cwd().resolve())

    input_csv = _resolve_input_csv()
    input_md5 = _md5_file(input_csv)
    print("input_csv:", input_csv)
    print("input_md5:", input_md5)

    _section("load + validate")
    df = pd.read_csv(input_csv)
    print("rows x cols:", df.shape)
    print("columns   :", list(df.columns))

    req = [CFG["text_col"], CFG["quarter_col"]]
    missing = [c for c in req if c not in df.columns]
    if missing:
        raise KeyError(f"missing required columns: {missing}")

    id_col = _pick_id_col(df)
    if id_col:
        df["sentence_id"] = df[id_col].astype(str)
    else:
        df["sentence_id"] = _build_stable_sentence_id(df, CFG["text_col"])

    df["belief_sentence"] = df[CFG["text_col"]].astype(str).str.strip()
    df["year_quarter"] = df[CFG["quarter_col"]].astype(str).str.strip()

    missing_text_rows = int((df["belief_sentence"].str.len() == 0).sum())
    sentence_id_list_hash = hashlib.md5(("|".join(df["sentence_id"].tolist())).encode("utf-8")).hexdigest()

    _section("embed")
    device = _pick_device()
    model_name = CFG["baseline_model_name"]
    print("model :", model_name)
    print("device:", device)

    model = SentenceTransformer(model_name, device=device)
    emb = model.encode(
        df["belief_sentence"].tolist(),
        batch_size=int(CFG["batch_size"]),
        show_progress_bar=True,
        device=device,
    )
    emb = np.asarray(emb, dtype="float32")
    emb_dim = int(emb.shape[1])

    norms = np.linalg.norm(emb, axis=1)
    norm_stats = {
        "min": float(norms.min()),
        "mean": float(norms.mean()),
        "max": float(norms.max()),
    }

    _section("export")
    out_df = df[["sentence_id", "year_quarter", "belief_sentence"]].copy()
    out_df["embedding"] = emb.tolist()

    emb_path = out_dir / "part1_baseline_embeddings.parquet"
    if str(CFG["save_format"]).lower() == "parquet":
        out_df.to_parquet(emb_path, index=False)
    else:
        emb_path = emb_path.with_suffix(".csv")
        out_df.to_csv(emb_path, index=False)

    summary = pd.DataFrame([{
        "embedding_set": "baseline",
        "input_csv": str(input_csv),
        "input_md5": input_md5,
        "input_rows": int(df.shape[0]),
        "missing_text_rows": missing_text_rows,
        "embedding_dim": emb_dim,
        "model_name": model_name,
        "device": device,
        "elapsed_seconds": round(time.time() - t0, 3),
        "embedding_norm_min": round(norm_stats["min"], 6),
        "embedding_norm_mean": round(norm_stats["mean"], 6),
        "embedding_norm_max": round(norm_stats["max"], 6),
        "sentence_id_list_hash": sentence_id_list_hash,
    }])
    summary_path = out_dir / "part1_baseline_summary.csv"
    summary.to_csv(summary_path, index=False)

    manifest = {
        "part": "part1_baseline",
        "embedding_set": "baseline",
        "run_timestamp": datetime.now().isoformat(timespec="seconds"),
        "input": {
            "input_csv_path": str(input_csv),
            "input_md5": input_md5,
            "input_row_count": int(df.shape[0]),
            "required_columns": req,
            "sentence_id_source": id_col if id_col else "generated_from_text_md5",
            "sentence_id_list_hash": sentence_id_list_hash,
        },
        "model": {
            "model_name": model_name,
            "device": device,
            "embedding_dim": emb_dim,
            "batch_size": int(CFG["batch_size"]),
        },
        "outputs": [
            {"name": "baseline_embeddings", "relpath": str(emb_path.relative_to(out_root))},
            {"name": "baseline_summary", "relpath": str(summary_path.relative_to(out_root))},
        ],
    }
    manifest_path = out_dir / "part1_baseline_embedding_manifest.json"
    with manifest_path.open("w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    # copy manifest to central manifests folder (stable retrieval)
    with (manifests_dir / manifest_path.name).open("w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    _section("console exhibit")
    print("outputs:")
    print("  embeddings:", emb_path)
    print("  summary   :", summary_path)
    print("  manifest  :", manifest_path)

    print("\nsummary preview (top 1):")
    print(summary.to_string(index=False))

    print("\nembeddings preview (top 10):")
    preview = out_df.head(10).copy()
    preview["embedding"] = preview["embedding"].apply(lambda v: f"len={len(v)}")
    print(preview.to_string(index=False))

    _header("part 1 completed")


# run
part1_baseline_embedding()



>>> ist 691 — phase 2 (part 1): baseline embedding

--------------------------------------------------------------------------------
[ inputs ]
--------------------------------------------------------------------------------
timestamp: 2025-12-12 12:17:34
platform : macOS-26.1-arm64-arm-64bit
python   : 3.12.7
cwd      : /Users/DJ/Dropbox/ADS_MA/IST.707/OLD_IST707-2025-SPR/Project-707/Final project delivery
input_csv: /Users/DJ/Dropbox/ADS_MA/IST.707/OLD_IST707-2025-SPR/Project-707/Final project delivery/belief_extraction_results_deduped_year_quarter_flattened_repeat_deduped.csv
input_md5: a6884e55e71b0fd3727a8f16966dd683

--------------------------------------------------------------------------------
[ load + validate ]
--------------------------------------------------------------------------------
rows x cols: (49325, 7)
columns   : ['hash_id', 'belief_sentence', 'year_quarter', 'strategic_focus', 'temporal_framing', 'certainty_level', 'market_position']

-------------------------

Batches:   0%|          | 0/771 [00:00<?, ?it/s]


--------------------------------------------------------------------------------
[ export ]
--------------------------------------------------------------------------------

--------------------------------------------------------------------------------
[ console exhibit ]
--------------------------------------------------------------------------------
outputs:
  embeddings: /Users/DJ/Dropbox/ADS_MA/IST.707/OLD_IST707-2025-SPR/Project-707/Final project delivery/IST691_phase2_output/part1_baseline/part1_baseline_embeddings.parquet
  summary   : /Users/DJ/Dropbox/ADS_MA/IST.707/OLD_IST707-2025-SPR/Project-707/Final project delivery/IST691_phase2_output/part1_baseline/part1_baseline_summary.csv
  manifest  : /Users/DJ/Dropbox/ADS_MA/IST.707/OLD_IST707-2025-SPR/Project-707/Final project delivery/IST691_phase2_output/part1_baseline/part1_baseline_embedding_manifest.json

summary preview (top 1):
embedding_set                                                                                 